# Metric — LLM-as-a-Judge Personalisation Success Rate (PSR)

$$\mathrm{PSR} = \frac{1}{N}\sum_{i} s_i, \qquad s_i = J\left(y^{+}_i, \hat{y}_i\right) \in \{0, 1\}$$

An external judge decides, per example, whether the generated answer satisfies
the **same implicit user preference** as the reference answer.

`JUDGE_MODEL = "deepseek-v4-pro"`, `JUDGE_TEMPERATURE = 0.0`,
`JUDGE_N_RUNS = 3` with per-run seeds; the reported label is the
**majority vote** over the three runs.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [2]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    get_ipython().run_line_magic("pip", "-q install -U pandas openai tqdm")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "subset_0"

OUTPUT_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_llm_judge_pref_deepseek"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Output dir:", OUTPUT_DIR.resolve())

Mounted at /content/drive
Subset: subset_0
Output dir: /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref


In [ ]:
SUBSET_DIR = PP_ROOT / SUBSET_NAME
SOURCE_CONFIGS = {
    "centralized_sft": {
        "results_dir": SUBSET_DIR / "v2_personamem_centralized_sft_snippet",
    },
    "centralized_orpo": {
        "results_dir": SUBSET_DIR / "v2_personamem_centralized_orpo_snippet",
    },
    "centralized_ctx_contrast_orpo": {
        "results_dir": SUBSET_DIR / "v2_personamem_centralized_ctx_contrast_orpo_snippet",
    },
    "centralized_ctx_contrast_neg_orpo": {
        "results_dir": SUBSET_DIR / "v2_personamem_centralized_ctx_contrast_negans_orpo_snippet",
    },
    "federated_sft": {
        "results_dir": SUBSET_DIR / "v2_personamem_federated_sft_snippet",
    },
    "federated_orpo": {
        "results_dir": SUBSET_DIR / "v2_personamem_federated_orpo_snippet",
    },
    "federated_ctx_contrast_orpo_gradavg": {
        "results_dir": SUBSET_DIR / "v2_personamem_federated_ctx_contrast_orpo_gradavg_snippet",
    },
}

ACTIVE_SOURCE = None

SELECT_PERSONAS: list | None = None
SPLITS = ["val"]

JUDGE_MODEL = "deepseek-v4-pro"
JUDGE_BATCH_SIZE = 4

JUDGE_TEMPERATURE = 0.0
JUDGE_SEED = 42
DISABLE_THINKING = True
JUDGE_N_RUNS = 3
JUDGE_VARY_SEED_PER_RUN = True

print("Judge provider: deepseek | model:", JUDGE_MODEL, "| seed:", JUDGE_SEED, "| temp:", JUDGE_TEMPERATURE, "| n_runs:", JUDGE_N_RUNS)
print("Active source:", ACTIVE_SOURCE or f"all ({len(SOURCE_CONFIGS)})")
print("Configured sources:")
for k, cfg in SOURCE_CONFIGS.items():
    print(f"  {k}: {cfg['results_dir']}")

In [6]:
import ast
import json
import pandas as pd

def load_predictions(results_dir, split):
    frames = []
    for csv_path in sorted(results_dir.glob(f"persona_*/{split}_predictions.csv")):
        df = pd.read_csv(csv_path)
        df["persona_id"] = csv_path.parent.name.replace("persona_", "")
        df["split"] = split
        frames.append(df)
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


def filter_personas(df, persona_ids):
    if persona_ids is None:
        return df
    keep = set(persona_ids)
    return df[df["persona_id"].astype(str).isin(keep)].reset_index(drop=True)


sources_to_run = (
    {ACTIVE_SOURCE: SOURCE_CONFIGS[ACTIVE_SOURCE]}
    if ACTIVE_SOURCE
    else SOURCE_CONFIGS
)

loaded: dict[str, dict[str, pd.DataFrame]] = {}
for name, cfg in sources_to_run.items():
    rd = cfg["results_dir"]
    if not rd.exists():
        print(f"[skip] {name}: {rd} not found")
        continue
    persona_filter = [str(p) for p in SELECT_PERSONAS] if SELECT_PERSONAS else None
    loaded[name] = {}
    for split in SPLITS:
        df = load_predictions(rd, split)
        if df is None:
            print(f"  [skip] {name} {split}: no {split}_predictions.csv in {rd}")
            continue
        loaded[name][split] = filter_personas(df, persona_filter)
        print(f"  {name} {split}: {len(loaded[name][split])} rows")
    if not loaded[name]:
        del loaded[name]

if not loaded:
    raise RuntimeError("No source data loaded.")

  centralized_sft val: 230 rows
  centralized_orpo val: 230 rows
  federated_gradavg_orpo val: 230 rows
  centralized_ctx_orpo val: 230 rows
  centralized_no_ctx_orpo val: 230 rows
  [skip] centralized_ctx_only_orpo val: no val_predictions.csv in /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_centralized_ctx_contrast_or_only_snippet


In [7]:
import math
import statistics
from openai import OpenAI
from tqdm.auto import tqdm

JUDGE_PROMPT = (
    "You are an expert evaluator for personalised assistant responses.\n"
    "Given the preference type, the ground-truth personalised answer, and a model output, "
    "decide whether the model output satisfies the SAME implicit user preference as the "
    "ground-truth answer.\n\n"
    "Preference type: {pref_type}\n"
    "Ground-truth answer: {correct_answer}\n"
    "Model output: {generated_answer}\n\n"
    "Does the model output satisfy the same preference as the ground-truth answer?\n"
    "Answer with a single token: 0 (no) or 1 (yes)."
)

def build_judge_messages(row):
    pref_type = str(row.get("pref_type", "unknown"))
    correct = str(row.get("correct_answer", ""))
    generated = str(row.get("generated_answer", ""))
    user_content = JUDGE_PROMPT.format(
        pref_type=pref_type,
        correct_answer=correct,
        generated_answer=generated,
    )
    return [{"role": "user", "content": user_content}]

def _softmax_pair(logprob_0, logprob_1, raw_token):
    if logprob_0 is not None and logprob_1 is not None:
        m = max(logprob_0, logprob_1)
        e0 = math.exp(logprob_0 - m)
        e1 = math.exp(logprob_1 - m)
        total = e0 + e1
        return e0 / total, e1 / total
    if logprob_1 is not None:
        p1 = math.exp(logprob_1)
        return 1.0 - p1, p1
    if logprob_0 is not None:
        p0 = math.exp(logprob_0)
        return p0, 1.0 - p0
    if raw_token == "1":
        return 0.0, 1.0
    if raw_token == "0":
        return 1.0, 0.0
    return 0.5, 0.5

class PreferenceJudge:
    def __init__(self, model_name="deepseek-v4-pro"):
        self.model_name = model_name
        api_key = os.environ["DEEPSEEK_API_KEY"]
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

    def _score_one(self, row, seed=None):
        messages = build_judge_messages(row)
        use_seed = JUDGE_SEED if seed is None else seed
        create_kwargs = dict(
            model=self.model_name,
            messages=messages,
            max_tokens=1,
            temperature=JUDGE_TEMPERATURE,
            seed=use_seed,
            logprobs=True,
            top_logprobs=20,
        )
        if DISABLE_THINKING:
            create_kwargs["extra_body"] = {"thinking": {"type": "disabled"}}
        try:
            resp = self.client.chat.completions.create(**create_kwargs)
        except Exception:
            create_kwargs.pop("extra_body", None)
            resp = self.client.chat.completions.create(**create_kwargs)
        choice = resp.choices[0]
        raw_token = (choice.message.content or "").strip()

        logprob_0 = None
        logprob_1 = None
        if choice.logprobs and choice.logprobs.content:
            item = choice.logprobs.content[0]
            logprob_map = {item.token.strip(): item.logprob}
            if item.top_logprobs:
                for tl in item.top_logprobs:
                    logprob_map[tl.token.strip()] = tl.logprob
            logprob_0 = logprob_map.get("0")
            logprob_1 = logprob_map.get("1")

        p0, p1 = _softmax_pair(logprob_0, logprob_1, raw_token)
        token = "1" if p1 >= p0 else "0"
        out = dict(row)
        out["judge_token"] = token
        out["judge_prob_0"] = p0
        out["judge_prob_1"] = p1
        out["judge_satisfies_pref"] = int(token == "1")
        out["judge_seed"] = use_seed
        out["judge_system_fingerprint"] = getattr(resp, "system_fingerprint", None)
        return out

    def judge_row_repeated(self, row):
        """Judge one row JUDGE_N_RUNS times; keep every run plus per-row mean/std."""
        runs = []
        for i in range(JUDGE_N_RUNS):
            seed = (JUDGE_SEED + i) if JUDGE_VARY_SEED_PER_RUN else JUDGE_SEED
            runs.append(self._score_one(row, seed=seed))
        out = dict(row)
        probs1 = [r["judge_prob_1"] for r in runs]
        probs0 = [r["judge_prob_0"] for r in runs]
        sats = [r["judge_satisfies_pref"] for r in runs]
        for i, r in enumerate(runs, start=1):
            out[f"judge_token_run{i}"] = r["judge_token"]
            out[f"judge_prob_0_run{i}"] = r["judge_prob_0"]
            out[f"judge_prob_1_run{i}"] = r["judge_prob_1"]
            out[f"judge_satisfies_pref_run{i}"] = r["judge_satisfies_pref"]
            out[f"judge_seed_run{i}"] = r["judge_seed"]
            out[f"judge_system_fingerprint_run{i}"] = r["judge_system_fingerprint"]
        _std = lambda xs: float(statistics.stdev(xs)) if len(xs) > 1 else 0.0
        out["n_runs"] = JUDGE_N_RUNS
        out["judge_prob_1_mean"] = float(statistics.mean(probs1))
        out["judge_prob_1_std"] = _std(probs1)
        out["judge_prob_0_mean"] = float(statistics.mean(probs0))
        out["judge_prob_0_std"] = _std(probs0)
        out["judge_satisfies_pref_rate"] = float(statistics.mean(sats))
        out["judge_satisfies_pref_std"] = _std(sats)
        out["judge_prob_1"] = out["judge_prob_1_mean"]
        out["judge_prob_0"] = out["judge_prob_0_mean"]
        out["judge_satisfies_pref"] = int(out["judge_satisfies_pref_rate"] >= 0.5)
        out["judge_token"] = "1" if out["judge_satisfies_pref"] else "0"
        return out

    def judge_batch(self, rows):
        return [self.judge_row_repeated(row) for row in rows]

    def judge_dataframe(self, df, desc):
        rows = [r.to_dict() for _, r in df.iterrows()]
        records = []
        for start in tqdm(range(0, len(rows), JUDGE_BATCH_SIZE), desc=desc):
            batch = rows[start:start + JUDGE_BATCH_SIZE]
            records.extend(self.judge_batch(batch))
        return pd.DataFrame(records)
judge = PreferenceJudge(JUDGE_MODEL)
print(f"Using DeepSeek judge: {JUDGE_MODEL} | seed={JUDGE_SEED} | temp={JUDGE_TEMPERATURE} | "
      f"thinking={'off' if DISABLE_THINKING else 'on'}")

Using OpenAI judge: gpt-4.1


In [ ]:
RUN_DETERMINISM_CHECK = True

if RUN_DETERMINISM_CHECK:
    _test_row = {
        "pref_type": "tone",
        "correct_answer": "Sure - here is a warm, encouraging note you can send your team.",
        "generated_answer": "Here is a friendly, upbeat message for your team.",
    }
    _runs = [judge._score_one(dict(_test_row)) for _ in range(3)]
    for i, r in enumerate(_runs):
        print(f"run {i}: token={r['judge_token']} p1={r['judge_prob_1']:.6f} fingerprint={r.get('judge_system_fingerprint')}")
    _p1 = [r['judge_prob_1'] for r in _runs]
    print(f"max |p1 - p1| spread = {max(_p1) - min(_p1):.2e} | tokens identical: {len({r['judge_token'] for r in _runs}) == 1}")

In [8]:
judged: dict[str, dict[str, pd.DataFrame]] = {}

for source_name, splits in loaded.items():
    judged[source_name] = {}
    for split in SPLITS:
        df = splits[split]
        if df.empty:
            continue
        if "generated_answer" not in df.columns:
            raise ValueError(f"{source_name}/{split}: missing generated_answer column")
        out = judge.judge_dataframe(df, desc=f"judge {source_name}/{split}")
        out_path = OUTPUT_DIR / f"{source_name}_{split}_judge.csv"
        out.to_csv(out_path, index=False)
        judged[source_name][split] = out
        sat_rate = out["judge_satisfies_pref"].mean()
        mean_p1 = out["judge_prob_1"].mean()
        print(f"{source_name} {split}: satisfaction rate={sat_rate:.4f} | mean judge_prob_1={mean_p1:.4f}")
        print(f"  saved {out_path}")

judge centralized_sft/val:   0%|          | 0/58 [00:00<?, ?it/s]

centralized_sft val: satisfaction rate=0.5957 | mean judge_prob_1=0.5954
  saved /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref/centralized_sft_val_judge.csv


judge centralized_orpo/val:   0%|          | 0/58 [00:00<?, ?it/s]

centralized_orpo val: satisfaction rate=0.6043 | mean judge_prob_1=0.6073
  saved /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref/centralized_orpo_val_judge.csv


judge federated_gradavg_orpo/val:   0%|          | 0/58 [00:00<?, ?it/s]

federated_gradavg_orpo val: satisfaction rate=0.5696 | mean judge_prob_1=0.5677
  saved /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref/federated_gradavg_orpo_val_judge.csv


judge centralized_ctx_orpo/val:   0%|          | 0/58 [00:00<?, ?it/s]

centralized_ctx_orpo val: satisfaction rate=0.6304 | mean judge_prob_1=0.6238
  saved /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref/centralized_ctx_orpo_val_judge.csv


judge centralized_no_ctx_orpo/val:   0%|          | 0/58 [00:00<?, ?it/s]

centralized_no_ctx_orpo val: satisfaction rate=0.5304 | mean judge_prob_1=0.5282
  saved /content/drive/MyDrive/privacy_perserving_pllm/subset_0/v2_personamem_llm_judge_pref/centralized_no_ctx_orpo_val_judge.csv


In [26]:
summary_rows = []
for source_name, splits in judged.items():
    for split, df in splits.items():
        summary_rows.append({
            "subset": SUBSET_NAME,
            "source": source_name,
            "split": split,
            "persona_id": "ALL",
            "n": len(df),
            "judge_satisfaction_rate": df["judge_satisfies_pref"].mean(),
            "mean_judge_prob_1": df["judge_prob_1"].mean(),
            "mean_judge_prob_0": df["judge_prob_0"].mean(),
            "mean_judge_prob_1_run_std": df["judge_prob_1_std"].mean() if "judge_prob_1_std" in df else float("nan"),
            "mean_judge_satisfies_run_std": df["judge_satisfies_pref_std"].mean() if "judge_satisfies_pref_std" in df else float("nan"),
        })
        for pid, g in df.groupby("persona_id"):
            summary_rows.append({
                "subset": SUBSET_NAME,
                "source": source_name,
                "split": split,
                "persona_id": pid,
                "n": len(g),
                "judge_satisfaction_rate": g["judge_satisfies_pref"].mean(),
                "mean_judge_prob_1": g["judge_prob_1"].mean(),
                "mean_judge_prob_0": g["judge_prob_0"].mean(),
                "mean_judge_prob_1_run_std": g["judge_prob_1_std"].mean() if "judge_prob_1_std" in g else float("nan"),
                "mean_judge_satisfies_run_std": g["judge_satisfies_pref_std"].mean() if "judge_satisfies_pref_std" in g else float("nan"),
            })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "judge_summary.csv", index=False)

overall = summary[summary["persona_id"] == "ALL"].drop(columns=["persona_id"]).reset_index(drop=True)
overall.to_csv(OUTPUT_DIR / "judge_overall_summary.csv", index=False)

per_persona = summary[summary["persona_id"] != "ALL"].reset_index(drop=True)
per_persona.to_csv(OUTPUT_DIR / "judge_per_persona_summary.csv", index=False)

print("Saved overall ->", OUTPUT_DIR / "judge_overall_summary.csv")
print("Saved per-persona ->", OUTPUT_DIR / "judge_per_persona_summary.csv")
print("\n=== Overall (per model) satisfaction on", SUBSET_NAME, "===")
print(overall[["source", "split", "n", "judge_satisfaction_rate"]].round(4).to_string(index=False))

with open(OUTPUT_DIR / "judge_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "judge_provider": "deepseek",
        "judge_model": JUDGE_MODEL,
        "judge_seed": JUDGE_SEED,
        "judge_temperature": JUDGE_TEMPERATURE,
        "disable_thinking": DISABLE_THINKING,
        "judge_n_runs": JUDGE_N_RUNS,
        "vary_seed_per_run": JUDGE_VARY_SEED_PER_RUN,
        "subset": SUBSET_NAME,
        "sources": list(judged.keys()),
        "source_dirs": {k: str(SOURCE_CONFIGS[k]["results_dir"]) for k in judged.keys()},
        "select_personas": SELECT_PERSONAS,
    }, f, indent=2)

                 source split persona_id   n  judge_satisfaction_rate  mean_judge_prob_1  mean_judge_prob_0
       centralized_orpo   val        ALL 230                   0.5696             0.5767             0.4233
       centralized_orpo   val        105   5                   0.4000             0.5004             0.4996
       centralized_orpo   val        148   4                   0.7500             0.7416             0.2584
       centralized_orpo   val        160   4                   0.5000             0.4548             0.5452
       centralized_orpo   val         18   5                   0.4000             0.4287             0.5713
       centralized_orpo   val        192   4                   0.5000             0.5000             0.5000
       centralized_orpo   val        194   5                   0.4000             0.4065             0.5935
       centralized_orpo   val        200   5                   0.8000             0.7633             0.2367
       centralized_orpo   va

In [27]:
for source_name, splits in judged.items():
    for split, df in splits.items():
        if "pref_type" not in df.columns:
            continue
        by_pref = df.groupby("pref_type").agg(
            n=("judge_satisfies_pref", "count"),
            satisfaction_rate=("judge_satisfies_pref", "mean"),
            mean_prob_1=("judge_prob_1", "mean"),
        ).sort_values("n", ascending=False)
        print(f"\n=== {source_name} / {split} by pref_type ===")
        print(by_pref.head(15).round(4).to_string())
        by_pref.to_csv(OUTPUT_DIR / f"{source_name}_{split}_judge_by_pref_type.csv")


=== centralized_orpo / val by pref_type ===
                                n  satisfaction_rate  mean_prob_1
pref_type                                                        
anti_stereotypical_pref        46             0.7174       0.7210
ask_to_forget                  40             0.5500       0.5405
therapy_background             35             0.4571       0.4707
neutral_preferences            32             0.6562       0.6316
sensitive_info                 27             0.7407       0.7745
health_and_medical_conditions  25             0.3600       0.3956
stereotypical_pref             25             0.4000       0.4144

=== centralized_ctx_orpo / val by pref_type ===
                                n  satisfaction_rate  mean_prob_1
pref_type                                                        
anti_stereotypical_pref        46             0.7826       0.7382
ask_to_forget                  40             0.6250       0.6282
therapy_background             35             0.